# Тарифы по ОГРН (клиенты с фото)

Минимальная тетрадка для проверки:
1. Импорты
2. Подключение к Impala
3. Список клиентов (ОГРН)
4. Поиск `tariff_name` по OGRN через R2-цепочку

In [ ]:
import re

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 240)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## Список клиентов (из фото)

In [ ]:
clients_from_photo = [
    {'client_name_photo': 'БИКМЕТОВА САЗИДА АСХАТОВНА (ИП)', 'ogrn_photo': '325028000279052'},
    {'client_name_photo': 'ЛАРИН АЛЕКСАНДР НИКОЛАЕВИЧ (ИП ГКФХ)', 'ogrn_photo': '304025512800036'},
    {'client_name_photo': 'БАГАУТДИНОВ РАБИЛ ЗИЯФОВИЧ (ИП)', 'ogrn_photo': '326028000007223'},
    {'client_name_photo': 'ЛИ ЖЭНЬШУНЬ (ИП)', 'ogrn_photo': '320028000097857'},
    {'client_name_photo': 'ООО "НЬЮРИС"', 'ogrn_photo': '1149102120441'},
    {'client_name_photo': 'ФРАНТЕНКО ВИКТОРИЯ КОНСТАНТИНОВНА (ИП)', 'ogrn_photo': '323385000127952'},
    {'client_name_photo': 'ООО "КРОСС+"', 'ogrn_photo': '1053811141230'},
    {'client_name_photo': 'ООО "САНИВА"', 'ogrn_photo': '1114613000490'},
    {'client_name_photo': 'ООО "СВТ"', 'ogrn_photo': '1095321000070'},
    {'client_name_photo': 'ООО "КУПЕЦ"', 'ogrn_photo': '1105543001520'},
    {'client_name_photo': 'КИЯН МАРИНА НИКОЛАЕВНА (ИП)', 'ogrn_photo': '326547600049591'},
    {'client_name_photo': 'ЛАНГ ЭДУАРД ЭДУАРДОВИЧ (ИП)', 'ogrn_photo': '322547600045721'},
    {'client_name_photo': 'МУШЕНКО МАКСИМ АНАТОЛЬЕВИЧ (ИП)', 'ogrn_photo': '323547600047523'},
    {'client_name_photo': 'АЛЛАЗОВА СУДАБА ИЛЬЯС КЫЗЫ (ИП)', 'ogrn_photo': '324246800091492'},
    {'client_name_photo': 'ООО "ПРИВОЛЬНОЕ"', 'ogrn_photo': '1115658002020'},
    {'client_name_photo': 'МИНАЗКИРОВ КИРИЛЛ ЭДУАРДОВИЧ (ИП)', 'ogrn_photo': '325565800030598'},
    {'client_name_photo': 'ООО "САНФОРТ-СЕРВИС"', 'ogrn_photo': '1025801369076'},
    {'client_name_photo': 'ТУГАЕВА ЮЛИЯ ВАЛЕРЬЕВНА (ИП)', 'ogrn_photo': '307583611500025'},
    {'client_name_photo': 'ШАБАЛИН ДМИТРИЙ НИКОЛАЕВИЧ (ИП ГКФХ)', 'ogrn_photo': '325595800099432'},
    {'client_name_photo': 'ООО "ЛИОНА"', 'ogrn_photo': '1225900023315'},
    {'client_name_photo': 'СЕЛЕЗНЕВА МАРГАРИТА ФАНИЛЬЕВНА (ИП)', 'ogrn_photo': '326965800023753'},
    {'client_name_photo': 'ПОПОВ АЛЕКСАНДР ВЛАДИМИРОВИЧ (ИП ГКФХ)', 'ogrn_photo': '319665800113563'},
    {'client_name_photo': 'НЕСТЕРОВА МАРИНА АНДРЕЕВНА (ИП)', 'ogrn_photo': '322665800105220'},
    {'client_name_photo': 'ЗИННАТУЛЛИНА ГУЗАЛЬ ВАКЫФОВНА (ИП ГКФХ)', 'ogrn_photo': '312167520800136'},
    {'client_name_photo': 'ГИЗЕТДИНОВ ИЛЬНУР АЛЬБЕРТОВИЧ (ИП)', 'ogrn_photo': '326169000072080'},
    {'client_name_photo': 'СИБГАТУЛЛИНА АЛИЯ ИРЕКОВНА (ИП)', 'ogrn_photo': '320169000141554'},
    {'client_name_photo': 'ООО "АВТОДЕТАЛИ"', 'ogrn_photo': '1131651000250'},
    {'client_name_photo': 'АШИМЖОНОВ РАСУЛЖОН АРАБЖОНОВИЧ (ИП)', 'ogrn_photo': '313169000166854'},
]

clients_df = pd.DataFrame(clients_from_photo)
clients_df.insert(0, 'photo_row_id', range(1, len(clients_df) + 1))

clients_df['ogrn'] = (
    clients_df['ogrn_photo']
    .astype(str)
    .str.replace(r'\D+', '', regex=True)
)
clients_df = clients_df[clients_df['ogrn'].str.len() > 0].copy()

ogrn_values = sorted(clients_df['ogrn'].dropna().astype(str).unique().tolist())

print(f'Клиентов в списке: {len(clients_df):,}')
print(f'Уникальных ОГРН: {len(ogrn_values):,}')
clients_df[['photo_row_id', 'client_name_photo', 'ogrn']].head(30)

## Поиск tariff_name по OGRN (R2-цепочка)

In [ ]:
if not ogrn_values:
    raise ValueError('Список OGRN пуст. Проверь входные данные.')

ogrn_sql = ', '.join([f"'{x}'" for x in ogrn_values])

sql_ogrn_tariff = f"""
with corp_in as (
    select
        regexp_replace(trim(cast(c.c_register_gos_reg_num_rec as string)), '[^0-9]', '') as ogrn,
        cast(c.id as string) as cft_id
    from ods.scd1_z_cl_corp c
    where regexp_replace(trim(cast(c.c_register_gos_reg_num_rec as string)), '[^0-9]', '') in ({ogrn_sql})
),
agr_one as (
    select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.c_agr_number as string) as contract_number_alpha,
        row_number() over (
            partition by cast(a.abs_agr_id as string)
            order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
    from ods_alpha.scd1_agreements a
    where coalesce(a.ods_deleted_flg, '0') <> '1'
),
m as (
    select
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_contract as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        cast(m.d_start as timestamp) as d_start,
        cast(m.d_end as timestamp) as d_end,
        cast(m.c_row_status as string) as row_status
    from ods.scd1_z_r2_ip_merchants m
)
select
    ci.ogrn,
    m.agr_id,
    a.contract_number_alpha,
    m.c_tariff_plan,
    cast(tp.c_name as string) as tariff_name,
    m.d_start,
    m.d_end,
    m.row_status
from corp_in ci
left join m
    on m.cft_id = ci.cft_id
left join ods.scd1_z_r2_tariff_plan tp
    on cast(tp.id as string) = m.c_tariff_plan
left join agr_one a
    on a.agr_id = m.agr_id
   and a.rn = 1
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ogrn_tariff_raw_df = imp.fetch(sql_ogrn_tariff)

if ogrn_tariff_raw_df is None:
    ogrn_tariff_raw_df = pd.DataFrame(columns=['ogrn', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name', 'd_start', 'd_end', 'row_status'])

for c in ['ogrn', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name', 'row_status']:
    if c in ogrn_tariff_raw_df.columns:
        ogrn_tariff_raw_df[c] = ogrn_tariff_raw_df[c].astype(str).str.strip()


# Полный список связок OGRN -> agr_id -> tariff_name
ogrn_tariff_full_df = clients_df.merge(
    ogrn_tariff_raw_df,
    on='ogrn',
    how='left'
).sort_values(
    ['ogrn', 'd_start', 'd_end', 'agr_id'],
    ascending=[True, False, False, True]
).reset_index(drop=True)


# Последний срез: 1 запись на OGRN
latest_base_df = ogrn_tariff_full_df.dropna(subset=['ogrn']).copy()
latest_base_df = latest_base_df.sort_values(
    ['ogrn', 'd_start', 'd_end', 'agr_id'],
    ascending=[True, False, False, True]
)
ogrn_tariff_latest_df = latest_base_df.drop_duplicates(subset=['ogrn'], keep='first').reset_index(drop=True)


# QC
input_ogrn_cnt = len(set(ogrn_values))
found_ogrn_cnt = int(
    ogrn_tariff_full_df.dropna(subset=['agr_id']).groupby('ogrn')['agr_id'].nunique().gt(0).sum()
)
missing_ogrn_df = (
    clients_df[['ogrn']].drop_duplicates()
    .merge(
        ogrn_tariff_full_df[['ogrn', 'agr_id']].dropna(subset=['agr_id']).drop_duplicates(subset=['ogrn']),
        on='ogrn',
        how='left',
        indicator=True
    )
)
missing_ogrn_df = missing_ogrn_df[missing_ogrn_df['_merge'] == 'left_only'][['ogrn']].reset_index(drop=True)

multi_agr_by_ogrn_df = (
    ogrn_tariff_full_df.dropna(subset=['agr_id'])
    .groupby('ogrn', as_index=False)
    .agg(agr_cnt=('agr_id', 'nunique'))
)
multi_agr_by_ogrn_df = multi_agr_by_ogrn_df[multi_agr_by_ogrn_df['agr_cnt'] > 1].sort_values('agr_cnt', ascending=False)

print('QC по подтяжке tariff_name по OGRN:')
print(f'Уникальных ОГРН во входе: {input_ogrn_cnt:,}')
print(f'ОГРН, где найден хотя бы один agr_id: {found_ogrn_cnt:,}')
print(f'ОГРН без найденного agr_id: {len(missing_ogrn_df):,}')
print(f'ОГРН с несколькими agr_id: {len(multi_agr_by_ogrn_df):,}')

print('\nПолный список связок OGRN -> agr_id -> tariff_name:')
display(ogrn_tariff_full_df)

print('\nПоследний срез (1 запись на OGRN):')
display(ogrn_tariff_latest_df)

if not missing_ogrn_df.empty:
    print('\nОГРН без найденного agr_id:')
    display(missing_ogrn_df)

if not multi_agr_by_ogrn_df.empty:
    print('\nОГРН, у которых несколько agr_id:')
    display(multi_agr_by_ogrn_df)